In [1]:
import os
import sys
from pathlib import Path

# Run this notebook from the repo root (audio-clf)
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

os.environ["DATASETS_AUDIO_BACKEND"] = "soundfile"
os.environ["TORCHCODEC_QUIET"] = "1"

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from load_data import load

dataset = load()

# Pick the held-out split the model never trained on
TEST_SPLIT = next(
    (s for s in ("test", "validation", "val") if s in dataset),
    list(dataset.keys())[-1]   # fallback: last split
)

print("Splits:", list(dataset.keys()))
for k in dataset.keys():
    print(f"  {k}: {len(dataset[k])} rows, columns: {dataset[k].column_names}")
print(f"\nUsing '{TEST_SPLIT}' as the held-out test split.")


Loading dataset...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Splits: ['train', 'validation', 'test']
  train: 9072 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time', 'augmented']
  validation: 504 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time', 'augmented']
  test: 504 rows, columns: ['audio', 'text_description', 'transcribed_text', 'phonemes', 'duration', 'gender', 'age_category', 'emotion', 'lang', 'pitch', 'speech_monotony', 'speaking_rate', 'noise', 'sdr_noise', 'reverberation', 'category', 'source', 'speaker_id', 'original_name', 'start_time', 'end_time',

In [2]:
from inference import resolve_checkpoint, load_model, run_row_inference
from load_data import get_row, read_audio
from train import SAMPLE_RATE
import numpy as np
from IPython.display import display, Audio

ckpt_path = resolve_checkpoint()
print(f"Using checkpoint: {ckpt_path}")

model, emotion_encoder, gender_encoder, age_encoder, \
    id2emotion, id2gender, id2age, processor = load_model(ckpt_path)
print("Model ready.")


Using checkpoint: /home/fatikh/issai/audio-clf/models/finetune/best_model_finetuned.pt
Model ready.


In [3]:
def check_row(index: int, split: str | None = None):
    split = split or TEST_SPLIT
    row = get_row(split, index, dataset=dataset)

    audio_data, sample_rate = read_audio(row["audio"])
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)
    duration_sec = len(audio_data) / sample_rate
    audio_widget = Audio(data=audio_data, rate=sample_rate, normalize=True, embed=True)

    # ── Inference ──────────────────────────────────────────────────────────────
    results = run_row_inference(row, model, processor, id2emotion, id2gender, id2age)

    # ── Print summary ──────────────────────────────────────────────────────────
    sep = "─" * 56
    print(sep)
    print(f"  Row {index}  |  split: '{split}'  |  {duration_sec:.2f}s")
    transcript = row.get("transcribed_text") or row.get("text") or row.get("transcript") or "—"
    description = row.get("text_description", "")
    print(f"  Transcript : {transcript}")
    if description:
        print(f"  Description: {description}")
    print(sep)
    for task, gt_key in [("emotion", "emotion"), ("gender", "gender"), ("age", "age_category")]:
        r = results[task]
        gt = row.get(gt_key, "—")
        match = "✓" if r["pred"] == str(gt) else "✗"
        print(f"\n  {task.capitalize():<10} pred: {r['pred']} ({r['confidence']*100:.1f}%)  {match}"
              f"   gt: {gt}")
        print(f"             top-3: {[(lbl, f'{p*100:.1f}%') for lbl, p in r['top3']]}")
    print(f"\n{sep}")

    display(audio_widget)
    return audio_widget, results


In [19]:
# ── Change index to inspect any row from the test split ──
audio, results = check_row(index=284)

────────────────────────────────────────────────────────
  Row 284  |  split: 'test'  |  1.60s
  Transcript : Әкем байға берейін деп жатыр.
  Description: A young male speaker with a high pitch and a very monotone, fearful tone, speaking at a rapid pace. The audio is noisy with a high level of background interference and no reverberation.
────────────────────────────────────────────────────────

  Emotion    pred: fearful (99.7%)  ✓   gt: fearful
             top-3: [('fearful', '99.7%'), ('sad', '0.3%'), ('surprised', '0.1%')]

  Gender     pred: M (97.3%)  ✓   gt: M
             top-3: [('M', '97.3%'), ('F', '2.7%')]

  Age        pred: young (81.5%)  ✓   gt: young
             top-3: [('young', '81.5%'), ('adult', '17.2%'), ('senior', '0.8%')]

────────────────────────────────────────────────────────


In [5]:
# 83, 30, 32, 33, 53

In [13]:
# Rows with ground-truth emotion = sad
split = dataset[TEST_SPLIT]
sad_indices = [i for i in range(len(split)) if split[i]["emotion"] == "fearful"]
print(f"Rows with emotion gt = 'sad': {len(sad_indices)}")
for i in sad_indices:
    row = split[i]
    transcript = (row.get("transcribed_text") or row.get("text") or "")[:70]
    print(f"  {i}: gender={row.get('gender')}, age={row.get('age_category')} | {transcript}...")

Rows with emotion gt = 'sad': 72
  225: gender=F, age=young | Жоқ, мен сүйікті алжапқышымды жоғалтып алдым....
  226: gender=F, age=young | Санжар, Қайсар, көрдіңдер ме?...
  227: gender=F, age=young | Екі күннен кейін жер шарына қауіп төнеді, жат жерліктен шабуыл жасайды...
  228: gender=F, age=young | Аяқ-қолыңды сезбей қаласың ғой....
  229: gender=F, age=young | Жарайсыңдар, қорқыныш жоқ па сендерде, біз бұл жерде ұтылып қаламыз ба...
  230: gender=F, age=young | Мен жалғыз өзім кілтті табамын, өзі арманыма жетем....
  231: gender=F, age=young | Бірдеме өртеніп жатқан шығар....
  232: gender=F, age=young | Мен қорықтым, себебі бұрын соңды мен коммуналдық шығындарын төлеп көрм...
  233: gender=F, age=young | Не, не, не, менің де кетуім керек, ей, ей, қай жаққа, қайда бара жатсы...
  279: gender=M, age=young | Ой, страшно, жоқ....
  280: gender=M, age=young | Маған болмайды, маған болмайды, маған болмайды....
  281: gender=M, age=young | Өлтірмейік пе ұстап алып?...
  282: gender=M, 